In [2]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
documents = [
    "Retrieval Augmented Generation (RAG) is a hybrid architecture that combines information retrieval with text generation models.",
    "RAG improves factual accuracy by grounding generated answers in retrieved documents.",
    "Failover retrieval systems use multiple search strategies when the first attempt fails.",
    "Strict retrieval uses a high similarity threshold to ensure precision.",
    "Broad retrieval lowers the threshold to improve recall.",
    "Query expansion improves recall by adding related terms to the original query.",
    "Hallucination occurs when a language model generates incorrect or fabricated information.",
    "Context-only synthesis ensures that answers are generated strictly from retrieved data.",
    "Out-of-domain queries refer to questions outside the knowledge base scope.",
    "Final refusal logic prevents misinformation."
]

In [4]:
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
doc_vectors = vectorizer.fit_transform(documents)

In [5]:
def retrieve(query, threshold):
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, doc_vectors)[0]
    results = []
    for i, score in enumerate(scores):
        if score >= threshold:
            results.append((documents[i], score))
    return results

In [6]:
def expand_query(query):
    expansions = {
        "failover": "multi stage retrieval failover strategy",
        "hallucination": "language model hallucination prevention grounding",
        "retrieval": "information retrieval cosine similarity tfidf"
    }
    for key in expansions:
        if key in query.lower():
            return expansions[key]
    return query + " detailed explanation"


In [7]:
def synthesize(context_docs, query):
    if not context_docs:
        return None
    combined_context = " ".join([doc for doc, _ in context_docs])
    answer = (
        "Answer generated strictly from retrieved context:\n\n"
        f"User Query: {query}\n\n"
        "Relevant Information:\n"
        f"{combined_context}\n\n"
        "This response is grounded only in the knowledge base."
    )
    return answer

In [8]:
def rag_failover_agent(query):

    print("\nUser Query:", query)
    results = retrieve(query, threshold=0.5)
    if results:
        print("Stage Triggered: Strict Retrieval (R1)")
        return synthesize(results, query)
    results = retrieve(query, threshold=0.25)
    if results:
        print("Stage Triggered: Broad Retrieval (R2)")
        return synthesize(results, query)
    expanded_query = expand_query(query)
    results = retrieve(expanded_query, threshold=0.25)
    if results:
        print("Stage Triggered: Query Expansion (R3)")

        print("Expanded Query:", expanded_query)
        return synthesize(results, query)
    print("Stage Triggered: Final Refusal (R5)")
    return "Sorry, I cannot answer this question based on the available knowledge base."

In [9]:
queries = [
    "What is RAG?",
    "Explain failover system",
    "How to prevent hallucination?",
    "Tell me about quantum physics"
]

for q in queries:
    print("\nAnswer:\n", rag_failover_agent(q))


User Query: What is RAG?
Stage Triggered: Final Refusal (R5)

Answer:
 Sorry, I cannot answer this question based on the available knowledge base.

User Query: Explain failover system
Stage Triggered: Query Expansion (R3)
Expanded Query: multi stage retrieval failover strategy

Answer:
 Answer generated strictly from retrieved context:

User Query: Explain failover system

Relevant Information:
Failover retrieval systems use multiple search strategies when the first attempt fails.

This response is grounded only in the knowledge base.

User Query: How to prevent hallucination?
Stage Triggered: Broad Retrieval (R2)

Answer:
 Answer generated strictly from retrieved context:

User Query: How to prevent hallucination?

Relevant Information:
Hallucination occurs when a language model generates incorrect or fabricated information.

This response is grounded only in the knowledge base.

User Query: Tell me about quantum physics
Stage Triggered: Final Refusal (R5)

Answer:
 Sorry, I cannot a